# Highlight Parameter Tuning Notebook

Use this notebook to tune `config.yaml` parameters so generated `highlights_metadata.json` better matches an expected metadata file.


## 1) Set inputs

- `VIDEO_PATH`: source match video
- `EXPECTED_METADATA_PATH`: expected `highlights_metadata.json`
- `BASE_CONFIG_PATH`: optional starting config (defaults to repo `config.yaml`)


In [ ]:
from __future__ import annotations

import itertools
import json
import math
import random
import shutil
import subprocess
import tempfile
from pathlib import Path

import yaml

REPO_ROOT = Path.cwd()
VIDEO_PATH = REPO_ROOT / 'sample_data' / 'match.mp4'
EXPECTED_METADATA_PATH = REPO_ROOT / 'sample_data' / 'expected_highlights_metadata.json'
BASE_CONFIG_PATH = REPO_ROOT / 'config.yaml'

MAX_EVALS = 40
RANDOM_SEED = 7

assert BASE_CONFIG_PATH.exists(), f'Missing config: {BASE_CONFIG_PATH}'
assert VIDEO_PATH.exists(), f'Missing video: {VIDEO_PATH}'
assert EXPECTED_METADATA_PATH.exists(), f'Missing expected metadata: {EXPECTED_METADATA_PATH}'

random.seed(RANDOM_SEED)
print('Inputs look good.')


## 2) Evaluation helpers


In [ ]:
def load_metadata(path: Path):
    data = json.loads(path.read_text(encoding='utf-8'))
    highlights = data.get('highlights', [])
    return sorted(highlights, key=lambda h: (h.get('start', 0.0), h.get('end', 0.0)))


def interval_iou(a_start, a_end, b_start, b_end):
    inter = max(0.0, min(a_end, b_end) - max(a_start, b_start))
    union = max(a_end, b_end) - min(a_start, b_start)
    if union <= 0:
        return 0.0
    return inter / union


def match_intervals(expected, predicted, iou_threshold=0.3):
    matched_expected = set()
    matched_predicted = set()
    iou_sum = 0.0

    for p_idx, p in enumerate(predicted):
        best_iou = 0.0
        best_e_idx = None
        for e_idx, e in enumerate(expected):
            if e_idx in matched_expected:
                continue
            iou = interval_iou(e['start'], e['end'], p['start'], p['end'])
            if iou > best_iou:
                best_iou = iou
                best_e_idx = e_idx
        if best_e_idx is not None and best_iou >= iou_threshold:
            matched_expected.add(best_e_idx)
            matched_predicted.add(p_idx)
            iou_sum += best_iou

    tp = len(matched_predicted)
    fp = max(0, len(predicted) - tp)
    fn = max(0, len(expected) - tp)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    avg_iou = iou_sum / tp if tp else 0.0

    return {
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'avg_iou': avg_iou,
    }


def clamp(value, low, high):
    return max(low, min(high, value))


def normalize_weights(raw_weights):
    keys = ['rally_duration', 'shot_count', 'ball_speed', 'movement', 'audio']
    values = [max(raw_weights[k], 1e-6) for k in keys]
    total = sum(values)
    return {k: v / total for k, v in zip(keys, values)}


expected = load_metadata(EXPECTED_METADATA_PATH)
print(f'Loaded {len(expected)} expected highlights')


## 3) Run one evaluation


In [ ]:
def build_config(base_cfg: dict, params: dict):
    cfg = json.loads(json.dumps(base_cfg))

    cfg.setdefault('highlight', {})
    cfg.setdefault('scoring', {})

    cfg['highlight']['score_threshold'] = float(clamp(params['score_threshold'], 0.05, 0.99))
    cfg['highlight']['merge_gap'] = float(clamp(params['merge_gap'], 0.0, 30.0))
    cfg['highlight']['clip_before'] = float(clamp(params['clip_before'], 0.0, 20.0))
    cfg['highlight']['clip_after'] = float(clamp(params['clip_after'], 0.0, 20.0))

    weights = normalize_weights(params['weights'])
    cfg['scoring'].update(weights)

    return cfg


def run_pipeline_with_config(config_data: dict, output_dir: Path):
    config_path = output_dir / 'candidate_config.yaml'
    config_path.write_text(yaml.safe_dump(config_data, sort_keys=False), encoding='utf-8')

    cmd = [
        'python',
        '-m',
        'pickleball_highlights.main',
        '--input',
        str(VIDEO_PATH),
        '--output',
        str(output_dir),
        '--config',
        str(config_path),
        '--log-level',
        'WARNING',
    ]
    result = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True)

    metadata_path = output_dir / 'highlights_metadata.json'
    if result.returncode != 0:
        return None, {'error': result.stderr[-1200:]}
    if not metadata_path.exists():
        return None, {'error': 'Pipeline finished but no highlights_metadata.json found'}

    predicted = load_metadata(metadata_path)
    return predicted, {'stderr_tail': result.stderr[-400:]}


def evaluate(params: dict, base_cfg: dict):
    tmp_dir = Path(tempfile.mkdtemp(prefix='tune_', dir=Path('/tmp')))
    try:
        cfg = build_config(base_cfg, params)
        predicted, meta = run_pipeline_with_config(cfg, tmp_dir)
        if predicted is None:
            return {'fitness': -1.0, 'metrics': {}, 'error': meta.get('error'), 'tmp_dir': str(tmp_dir)}

        metrics = match_intervals(expected, predicted, iou_threshold=0.3)
        fitness = 0.75 * metrics['f1'] + 0.25 * metrics['avg_iou']

        return {
            'fitness': fitness,
            'metrics': metrics,
            'predicted_count': len(predicted),
            'tmp_dir': str(tmp_dir),
            'config': cfg,
            'params': params,
        }
    except Exception as exc:
        return {'fitness': -1.0, 'metrics': {}, 'error': str(exc), 'tmp_dir': str(tmp_dir)}


In [ ]:
base_cfg = yaml.safe_load(BASE_CONFIG_PATH.read_text(encoding='utf-8'))

default_params = {
    'score_threshold': float(base_cfg.get('highlight', {}).get('score_threshold', 0.4)),
    'merge_gap': float(base_cfg.get('highlight', {}).get('merge_gap', 10.0)),
    'clip_before': float(base_cfg.get('highlight', {}).get('clip_before', 5.0)),
    'clip_after': float(base_cfg.get('highlight', {}).get('clip_after', 5.0)),
    'weights': {
        'rally_duration': float(base_cfg.get('scoring', {}).get('rally_duration', 0.35)),
        'shot_count': float(base_cfg.get('scoring', {}).get('shot_count', 0.25)),
        'ball_speed': float(base_cfg.get('scoring', {}).get('ball_speed', 0.15)),
        'movement': float(base_cfg.get('scoring', {}).get('movement', 0.15)),
        'audio': float(base_cfg.get('scoring', {}).get('audio', 0.10)),
    },
}

baseline = evaluate(default_params, base_cfg)
baseline


## 4) Search parameter combinations


In [ ]:
threshold_grid = [0.25, 0.35, 0.45, 0.55]
merge_gap_grid = [5.0, 10.0, 15.0]
clip_before_grid = [2.0, 5.0, 8.0]
clip_after_grid = [2.0, 5.0, 8.0]

weight_presets = [
    {'rally_duration': 0.35, 'shot_count': 0.25, 'ball_speed': 0.15, 'movement': 0.15, 'audio': 0.10},
    {'rally_duration': 0.40, 'shot_count': 0.25, 'ball_speed': 0.15, 'movement': 0.10, 'audio': 0.10},
    {'rally_duration': 0.30, 'shot_count': 0.30, 'ball_speed': 0.20, 'movement': 0.10, 'audio': 0.10},
    {'rally_duration': 0.30, 'shot_count': 0.20, 'ball_speed': 0.15, 'movement': 0.20, 'audio': 0.15},
]

search_space = list(itertools.product(
    threshold_grid,
    merge_gap_grid,
    clip_before_grid,
    clip_after_grid,
    range(len(weight_presets)),
))

random.shuffle(search_space)
search_space = search_space[:MAX_EVALS]

results = []
best = baseline

for i, (thr, gap, before, after, w_idx) in enumerate(search_space, start=1):
    params = {
        'score_threshold': thr,
        'merge_gap': gap,
        'clip_before': before,
        'clip_after': after,
        'weights': weight_presets[w_idx],
    }

    result = evaluate(params, base_cfg)
    results.append(result)

    if result['fitness'] > best['fitness']:
        best = result

    print(f"[{i:02d}/{len(search_space)}] fitness={result['fitness']:.4f} f1={result.get('metrics', {}).get('f1', 0):.4f} iou={result.get('metrics', {}).get('avg_iou', 0):.4f}")

best


## 5) Save best tuned config


In [ ]:
best_config_path = REPO_ROOT / 'tuned_config.yaml'
if best.get('config'):
    best_config_path.write_text(yaml.safe_dump(best['config'], sort_keys=False), encoding='utf-8')
    print(f'Wrote {best_config_path}')
    print('Best metrics:', best['metrics'])
else:
    print('No successful run yet; nothing written.')
